# Installing Spotipy

In [ ]:
##!conda install -c conda-forge 

## Loading credentials from the config file

#### Make sure that you have stored your spotify client_id and client_secret in a separate config.py file
#### Once that's done, we import it

## Starting with Spotify API

In [6]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import pandas as pd
import json
import os

# Import credentials from config.py (which is gitignored)
try:
    import config
    client_id = config.client_id
    client_secret = config.client_secret
except ImportError:
    raise ImportError(
        "config.py not found. Copy config.example.py to config.py "
        "and fill in your Spotify credentials."
    )

auth_manager = SpotifyClientCredentials(
    client_id=client_id,
    client_secret=client_secret
)

sp = spotipy.Spotify(auth_manager=auth_manager)
print("Spotify client initialized successfully.")

Spotify client initialized successfully.


In [ ]:
results = sp.search(q="Shape of You", type="track", limit=1)
results

# Understanding the json

In [ ]:
track = results["tracks"]["items"][0]

track_name = track["name"]
artist_name = track["artists"][0]["name"]
album_name = track["album"]["name"]
track_id = track["id"]

print(track_name, artist_name, album_name, track_id)

## Other Info

In [17]:
from IPython.display import IFrame

IFrame(src=f"https://open.spotify.com/embed/track/{track_id}",
       width="300",
       height="380")

## Getting the track_id

In [21]:
track = sp.search(q='track:Blinding Lights', type='track', limit=1)
track_id = track['tracks']['items'][0]['id']
print("Track ID:", track_id)

Track ID: 0VjIjW4GlUZAMYd2vXMi3b


## Embedded track player

In [19]:
from IPython.display import IFrame

IFrame(src=f"https://open.spotify.com/embed/track/{track_id}",
       width="300",
       height="380")

# Getting the Audio features of a song

## Building a Data frame of audio features

In [31]:
track_id = "3AJwUDP919kvQ9QcozQPxg"

In [ ]:
features = sp._get("audio-features/" + track_id)
df_features = pd.DataFrame([features])
df_features

In [ ]:
features = sp._get("audio-features/" + track_id)
features_list = [features]
features_list

In [ ]:
features_list = sp.audio_features([track_id])

features = features_list[0] if features_list and features_list[0] else None
features

## Searching a playlist

In [ ]:
playlists = sp.search(q='rock', type='playlist', limit=5)
for playlist in playlists['playlists']['items']:
    print(playlist['name'], '->', playlist['id'])

## Extracting a song from playlist

In [ ]:
# Grab one song from a playlist
playlist_id = "7DgPQwzEoUVfQYBiMLER9Z"   # example playlist ID from earlier cell

playlist = sp.playlist(playlist_id)
first_track = playlist['tracks']['items'][0]['track']

print("Track name:", first_track['name'])
print("Artists:", [a['name'] for a in first_track['artists']])
print("Track ID:", first_track['id'])

## Extracting the songs of a playlist

Pagination using "next".
When you collect songs from a playlist using sp.playlist_tracks, you're limited by the limit parameter, which has a maximum (and default) value of 100. When the playlist has more than 100 songs, you have to collect them by navigating through the "pages" of the results.

The parameter offset allows you to retrieve resuls starting at a certain position: if you start at position 101, you'd get the next "page" of results. An offset of 201 would give you the third page, and so on.

The function sp.next() does the same, but in a simpler way: it can be used on the results from any request to directly retrieve the results for the next page.

We can check whether there's a next page or not by accessing the key next on the results from any request.

In [ ]:
# Get ALL tracks from a playlist, following "next" pages
def get_playlist_tracks(playlist_id):
    all_tracks = []
    results = sp.playlist_tracks(playlist_id)
    all_tracks.extend(results['items'])

    while results['next']:
        results = sp.next(results)          # fetch next page
        all_tracks.extend(results['items'])

    return all_tracks

tracks = get_playlist_tracks("7DgPQwzEoUVfQYBiMLER9Z")
print("Total tracks:", len(tracks))
for t in tracks[:5]:
    tr = t['track']
    print(tr['name'], "-", ", ".join(a['name'] for a in tr['artists']))

### Chopping a big playlist into chunks

In [ ]:
def get_playlist_tracks_chunked(playlist_id, chunk_size=100):
    all_tracks = []
    offset = 0
    while True:
        results = sp.playlist_tracks(playlist_id, offset=offset, limit=chunk_size)
        items = results['items']
        if not items:
            break
        all_tracks.extend(items)
        offset += chunk_size
        if results['next'] is None:
            break
    return all_tracks

chunks = get_playlist_tracks_chunked("7DgPQwzEoUVfQYBiMLER9Z")
print("Total tracks (chunked):", len(chunks))

### Example with rock

In [ ]:
# Find a rock playlist and pull its tracks
playlists = sp.search(q='rock', type='playlist', limit=5)
for playlist in playlists['playlists']['items']:
    if playlist is None:      # guard against None entries
        continue
    print(playlist['name'], '->', playlist['id'])

# Pick the first valid one
rock_playlist = next(p for p in playlists['playlists']['items'] if p)
rock_tracks = get_playlist_tracks(rock_playlist['id'])
print("\nTracks in:", rock_playlist['name'])
for t in rock_tracks[:10]:
    if t['track']:
        print(" -", t['track']['name'])

## Optional(Extra)

## Getting the artists of the playlist 

In [ ]:
# Unique artists from a playlist
artist_names = set()
for item in tracks:                     # 'tracks' from earlier cell
    if item['track'] is None:
        continue
    for artist in item['track']['artists']:
        artist_names.add(artist['name'])

print("Unique artists:", len(artist_names))
for name in sorted(artist_names)[:20]:
    print(" -", name)

# Getting albums 

In this section we will work with albums to extract information. We will start by extracting all the albums of an artist.

In [ ]:
def get_artist_albums(artist_id, include_groups='album,single,compilation'):
    """Return all albums for an artist, handling pagination."""
    albums = []
    results = sp.artist_albums(artist_id, album_type=include_groups, limit=50)
    albums.extend(results['items'])
    while results['next']:
        results = sp.next(results)
        albums.extend(results['items'])
    return albums

# Example: Coldplay's artist ID
coldplay_id = "4gzpq5DPGxSnKTe4SA8HAU"
albums = get_artist_albums(coldplay_id)
print("Total albums:", len(albums))
for alb in albums[:10]:
    print(alb['name'], "->", alb['id'], "(", alb['release_date'], ")")

Example: Coldplay

In [ ]:
coldplay_id = "4gzpq5DPGxSnKTe4SA8HAU"
coldplay = sp.artist(coldplay_id)
print("Artist:", coldplay['name'])
print("Genres:", coldplay['genres'])
print("Followers:", coldplay['followers']['total'])

# Top tracks
top = sp.artist_top_tracks(coldplay_id, country='US')
for t in top['tracks'][:5]:
    print(" -", t['name'])

## Getting the songs of a given album

In [ ]:
def get_album_tracks(album_id):
    results = sp.album_tracks(album_id)
    tracks = results['items']
    while results['next']:
        results = sp.next(results)
        tracks.extend(results['items'])
    return tracks

# Pick an album from the Coldplay list and list its songs
example_album_id = albums[0]['id']
album_tracks = get_album_tracks(example_album_id)
print("Album:", albums[0]['name'])
for t in album_tracks:
    print(f"  {t['track_number']:>2}. {t['name']}")